In [16]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_xJ')

enss=['b','c','d','e']
# stouts=[5,7,10,13,15,20,25,30,35,40]
stouts=[5,7,10,13,15,20]

tfphys=[0.4,0.5,0.6,0.7,0.8,0.9,1.0]
tcphys=[0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5]
tftcphys=[(tfphy,tcphy) for tfphy in tfphys for tcphy in tcphys if tfphy>=2*tcphy]


ens2Njk={'b':725,'c':400,'d':493,'e':516}

js_conn=['j+;conn','j-;conn']
js_discq=['j+;disc','js;disc','jc;disc'] 
js_gluon=[f'jg;stout{nst}' for nst in stouts]
js_disc=js_discq+js_gluon

tftcphy_A20_conn=tftcphy_B20_conn=tftcphy_C20_conn=(0.8,0.2)
tftcphy_A20_discq=tftcphy_A20_gluon=tftcphy_B20_discq=tftcphy_B20_gluon=tftcphy_C20_discq=tftcphy_C20_gluon=(0.7,0.3)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
# # ej_key2bare_A20at0=yu.load_pkl_reg('ej_key2bare_A20at0')
# # ens='b'; j='j+;conn'
# # ej_key2bare_A20at0[(ens,j)].shape

# key2phy_A20=yu.load_pkl_reg('key2phy_A20')
# ens='b'
# print('u-d:\t\t',yu.jackme_un2str(key2phy_A20[(ens,'jv1')]))
# print('u+d(conn):\t',yu.jackme_un2str(key2phy_A20[(ens,'jq;conn')]))
# print('u+d:\t\t',yu.jackme_un2str(key2phy_A20[(ens,'ju')]+key2phy_A20[(ens,'jd')]))

In [32]:
# ej_key2bare_A20at0=yu.load_pkl_reg('ej_key2bare_A20at0')
# ej_key2bare_A20at0.keys()

In [2]:
path='data_aux/RCs.pkl'
with open(path,'rb') as f:
    ens2RCs_me=pickle.load(f)
ens2RCs={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs[ens][key]=yu.jackknife_pseudo(ens2RCs_me[ens][key],ens2RCs_me[ens][f'{key}_err']*0+1e-10,ens2Njk[ens])[:,0]
        
path='data_aux/RCs_np.pkl'
with open(path,'rb') as f:
    ens2RCs_np_me=pickle.load(f)
ens2RCs_np={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_np_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs_np[ens][key]=yu.jackknife_pseudo(ens2RCs_np_me[ens][key],ens2RCs_np_me[ens][f'{key}_err']+1e-10,ens2Njk[ens])[:,0]

In [3]:
for ens in enss:
    yu.setpath('analysis_avgx_2')
    j='j-;conn'; app='2st_True'
    fits=yu.getFits(f'{ens}_{j}_{app}')
    fitlabels=[fit[0] for fit in fits]
    
    tfphy=0.6; tcphy=0.2
    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    fit=fits[ind]
    print(ens, fit[0], (fit[0][0]*yu.ens2a[ens],fit[0][1]*yu.ens2a[ens]))

    yu.setpath('analysis_A20')
    n2qpp1=(1,1,0); ff='A20'; c1='all'
    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_2st')
    yu.setpath('analysis_xJ')
    fitlabels=[fit[0] for fit in fits]
    
    tfphy=0.6; tcphy=0.2
    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    fit=fits[ind]
    print(ens, fit[0], (fit[0][0]*yu.ens2a[ens],sum(fit[0][1])/2*yu.ens2a[ens]))

b (8, 3) (0.63584, 0.23843999999999999)
b (8, (2, 3)) (0.63584, 0.1987)
c (8, 3) (0.54552, 0.20457)
c (8, (3, 3)) (0.54552, 0.20457)
d (10, 4) (0.5685, 0.2274)
d (10, (3, 4)) (0.5685, 0.19897499999999999)
e (11, 4) (0.5381199999999999, 0.19568)
e (11, (5, 3)) (0.5381199999999999, 0.19568)


# A20

In [4]:
def run():
    cttejn_key2bare_A20={}

    yu.setpath('analysis_avgx_2')
    for ens in enss:
        for j in js_conn+js_disc:
            app='2st_True' if j in js_conn else 'const'
            fits=yu.getFits(f'{ens}_{j}_{app}')
            fitlabels=[fit[0] for fit in fits]

            for tfphy,tcphy in tftcphys:
                ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                fit=fits[ind]
                # if ens=='e':
                #     print(tfphy,tcphy,fit[0])
                
                if j in js_conn:
                    key=('equal',tfphy,tcphy,ens,j,(0,0,0))
                    cttejn_key2bare_A20[key]=fit[1][:,0]
                    key=('all',tfphy,tcphy,ens,j,(0,0,0))
                    cttejn_key2bare_A20[key]=fit[1][:,0]
                else:
                    key=('unequal',tfphy,tcphy,ens,j,(0,1,1))
                    cttejn_key2bare_A20[key]=fit[1][:,0]
                
    yu.setpath('analysis_A20')
    def decodeTask(task):
        n2qpp1,ff,j=task.split('_')
        j=j.replace(',',';')
        n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
        return (n2qpp1,ff,j)
    path='data_aux/dat_ignore/analysis_A20_conn_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_conn:
            for c1 in ['all','unequal','equal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_2st')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                                
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_A20[key]=fit[1][:,0]
                        
    path='data_aux/dat_ignore/analysis_A20_disc_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_disc:
            for c1 in ['unequal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_const')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_A20[key]=fit[1][:,0]
                
    yu.setpath('analysis_xJ')
    
    return cttejn_key2bare_A20

cttejn_key2bare_A20=yu.load_pkl_reg('cttejn_key2bare_A20')
if cttejn_key2bare_A20 is None: 
    cttejn_key2bare_A20=run()
    yu.save_pkl_reg('cttejn_key2bare_A20',cttejn_key2bare_A20)

In [17]:
def run():
    ej_key2bare_A20at0={}

    colors=['r','g','b','orange']
    for j in js_conn+js_disc:
    # for j in ['j-;conn']:
        c1s=['unequal','all','equal'] if j in js_conn else ['unequal']
        
        fig,axs=yu.getFigAxs(len(enss)+1,1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey=True)
        fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
        yu.addRowHeader(axs,enss)
        yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])
        
        tftcphy = tftcphy_A20_conn if j in js_conn else tftcphy_A20_discq if j in js_discq else tftcphy_A20_gluon if j in js_gluon else 1/0
        
        for axr in axs:
            for ax in axr:
                yu.addRefLine(ax,0)

        for ic1,c1 in enumerate(c1s):
            key2bare={cttejn[3:]:cttejn_key2bare_A20[cttejn] for cttejn in cttejn_key2bare_A20.keys() if cttejn[:3]==(c1,tftcphy[0],tftcphy[1])}
            for iens,ens in enumerate(enss):
                ax=axs[iens,0]            
                xunit=1; yunit=1
                
                if j in js_conn:
                    Zqq='Zqq^s' if j in ['j+;conn'] else 'Zqq'
                    mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                    Z=ens2RCs_me[ens][f'{Zqq}({mn})']
                else:
                    Z=1
                
                n2qpp1s=[key[-1] for key in key2bare.keys() if key[0]==ens and key[1]==j]
                Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
                ys=np.array([key2bare[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T
                
                if ic1==0:
                    if j in js_conn:
                        m,e=yu.jackme(cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]*ens2RCs_me[ens][f'{Zqq}(mu=nu)'])
                    else:
                        m,e=yu.jackme(cttejn_key2bare_A20[('unequal',tftcphy[0],tftcphy[1],ens,j,(0,1,1))])
                    axs[iens,-1].errorbar(-1,m,e,color='orange',label=yu.un2str(m,e))
                
                color=colors[ic1]
                
                mean,err=yu.jackme(ys*Z)
                plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
                axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
                axs[-1,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=colors[iens])
                
                
                if j in js_conn:
                    Q2cut=1
                    doFit=yu.doFit_dipole
                    fitfunc=yu.fitfunc_dipole
                else:
                    Q2cut=0.5
                    doFit=yu.doFit_linear
                    fitfunc=yu.fitfunc_linear
                inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
                Q2s=Q2s[inds]
                ys=ys[:,inds]
            
                pars_jk,chi2_jk,Ndof=doFit(Q2s,ys,corrQ=False)
                
                xs_plt=np.arange(0,Q2cut+1e-5,0.05)
                y_plt=np.array([fitfunc(xs_plt,*pars) for pars in pars_jk])
                mean,err=yu.jackme(y_plt*Z)
                plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                
                axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
                axs[iens,ic1+1].legend()
                
                axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
                axs[iens,-1].set_xlim([-5,5])
                axs[iens,-1].legend()
                
                if c1=='unequal' and j in ['j+;conn']:
                    t=y_plt[:,0]
                    ej_key2bare_A20at0[(ens,j)]=t
                if c1=='all' and j in ['j-;conn']:
                    factor=ens2RCs_np_me[ens][f'Zqq(mu=nu)']/ens2RCs_np_me[ens][f'Zqq(mu!=nu)']
                    t1=y_plt[:,0]*factor
                    t2=cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]*factor
                    print(yu.jackme_un2str(cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]))
                    print(yu.jackme_un2str(np.transpose([t1,t2,t1-t2])*ens2RCs_np_me[ens][f'Zqq(mu!=nu)']))
                    print(yu.formatMatrix(yu.cov2correlation(yu.jackmec(np.transpose([t1,t2]))[-1])))
                    print()
                    ej_key2bare_A20at0[(ens,j+'_0Q2')]=cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]*ens2RCs_np_me[ens][f'Zqq(mu=nu)']
                    ej_key2bare_A20at0[(ens,j)]=t1
                if j in ['j-;conn']:
                    mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                    t= y_plt[:,0] * ens2RCs_np_me[ens][f'Zqq({mn})']
                    ej_key2bare_A20at0[(ens,j+'_'+c1)]=t
                if c1=='unequal' and j in js_disc:
                    t=cttejn_key2bare_A20[('unequal',tftcphy[0],tftcphy[1],ens,j,(0,1,1))]
                    ej_key2bare_A20at0[(ens,j)]=t
        # axs[0,0].set_ylim([0.05,0.19])
        # axs[0,0].set_yticks(np.arange(0.06,0.19,0.02))
        yu.finalizePlot(f'Q2_mulc1_A20/{j}',mkdirQ=True,closeQ=True)
        
    return ej_key2bare_A20at0

# tftcphy_A20_conn=(0.6,0.2)
ej_key2bare_A20at0=yu.load_pkl_reg('ej_key2bare_A20at0')
if ej_key2bare_A20at0 is None or True:
    ej_key2bare_A20at0=run()
    yu.save_pkl_reg('ej_key2bare_A20at0',ej_key2bare_A20at0)

0.153(12)
[0.168(13), 0.171(14), -0.0030(80)]
       1   0.8224
  0.8224        1

0.133(12)
[0.162(13), 0.153(14), 0.0087(61)]
       1   0.9008
  0.9008        1

0.137(11)
[0.166(13), 0.161(13), 0.0042(54)]
       1   0.9144
  0.9144        1

0.1324(88)
[0.168(10), 0.160(11), 0.0082(56)]
       1   0.8572
  0.8572        1



In [18]:
fig,axs=yu.getFigAxs(1,1,Lrow=4,Lcol=8)
ax=axs[0,0]
xunit=1; yunit=1

c1='all'; j='j-;conn'
tftcphy = tftcphy_A20_conn 
key2bare={cttejn[3:]:cttejn_key2bare_A20[cttejn] for cttejn in cttejn_key2bare_A20.keys() if cttejn[:3]==(c1,tftcphy[0],tftcphy[1])}

y_fit=[]; Q2_fit=[]; a2_fit=[]
for iens,ens in enumerate(enss):
    color=yu.colors8[iens]; fmt=yu.fmts8[iens]
    n2qpp1s=[key[-1] for key in key2bare.keys() if key[0]==ens and key[1]==j]
    n2qpp1s=[n2qpp1 for n2qpp1 in n2qpp1s if yum.n2qpp12Q2(n2qpp1,ens)<=1]
    Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
    ys=np.array([key2bare[(ens,j,n2qpp1)][:]*ens2RCs[ens]['Zqq(mu=nu)'] for n2qpp1 in n2qpp1s]).T
    
    y_fit.append(ys); Q2_fit+=list(Q2s); a2_fit+=[yu.ens2a[ens]**2]*len(Q2s)
    
    mean,err=yu.jackme(ys)
    plt_x=(Q2s+iens*0.003)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt)

y_fit=yu.superjackknife(y_fit); Q2_fit=np.array(Q2_fit); a2_fit=np.array(a2_fit)

fitfunc_dipole_a2 = lambda Q2s,a2s,g,m,cg,cm: (g+cg*a2s)/(1+Q2s/(m+cm*a2s)**2)**2
fitfunc_dipole_a2 = lambda Q2s,a2s,g,m,cg,cm: g/(1+Q2s/m**2)**2 + cg*a2s + cm*a2s*Q2s
def fitfunc(pars):
    return fitfunc_dipole_a2(Q2_fit,a2_fit,*pars)

pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_fit,pars0=[0.16,1,0,0],mask='uncorrelated')

Q2_plt=np.arange(0,1,1/40)
y_jk=[fitfunc_dipole_a2(Q2_plt,0,*pars) for pars in pars_jk]
mean,err=yu.jackme(y_jk)

plt_x=(Q2_plt)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
ax.fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color='magenta',alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
ax.legend()

PDF_SET_1='JAM22-PDF_proton_nlo'
PDF_SET_2='NNPDF40_nnlo_as_01180'
j2me_1=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET_1}.pkl')
j2me_2=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET_2}.pkl')


mean,err=j2me_1['jv1']
plt_x=-0.02; plt_y=mean*yunit; plt_yerr=err*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color='black')
mean,err=j2me_2['jv1']
plt_x=-0.03; plt_y=mean*yunit; plt_yerr=err*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color='black')


yu.finalizePlot('j-;conn_dipole_a2')

print(yu.jackme_un2str(pars_jk))
print(np.max(Q2_fit*a2_fit))

PDF_SET_1='JAM22-PDF_proton_nlo'
PDF_SET_2='NNPDF40_nnlo_as_01180'



[0.168(16), 1.49(19), -0.8(3.9), 3.3(4.6)]
0.0061700339202082725


In [19]:
fig,axs=yu.getFigAxs(1,1,Lrow=6,Lcol=8)
ax=axs[0,0]; xunit=1; yunit=1
ax.set_xlim([0,0.008])
ax.set_ylim([0.13,0.23])

mean,err=j2me_1['jv1']
plt_x=0.0002; plt_y=mean*yunit; plt_yerr=err*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color='black')
mean,err=j2me_2['jv1']
plt_x=0.0003; plt_y=mean*yunit; plt_yerr=err*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color='black')

for ic1, c1 in enumerate(['equal','all','unequal','0Q2']):
    ens2dat={ens:ej_key2bare_A20at0[(ens,f'j-;conn_{c1}')] for ens in enss}
    color=yu.colors8[ic1]; fmt=yu.fmts8[ic1]
    for ens in enss:
        mean,err=yu.jackme(ens2dat[ens])
        plt_x=yu.ens2a[ens]**2+0.0001*ic1; plt_y=mean*yunit; plt_yerr=err*yunit
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt)
    fits=yu.doFits_continuumExtrapolation(ens2dat,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['const','linear'],supjackQ=True)
    pars_jk,probs_jk=yu.jackMA(fits)
    mean,err=yu.jackme(pars_jk)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1,label=f'{c1}={yu.un2str(mean[0],err[0])}')
        
ax.legend()
yu.finalizePlot('j-;conn_various_a2')

# B20

In [37]:
def run():
    cttejn_key2bare_B20={}

    yu.setpath('analysis_B20')
    def decodeTask(task):
        n2qpp1,ff,j=task.split('_')
        j=j.replace(',',';')
        n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
        return (n2qpp1,ff,j)
    path='data_aux/dat_ignore/analysis_B20_conn_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_conn:
            for c1 in ['all','unequal','equal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_2st')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_B20[key]=fit[1][:,0]
                        
    yu.setpath('analysis_B20_2')
    path='data_aux/dat_ignore/analysis_B20_disc_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_disc:
            for c1 in ['unequal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_const')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_B20[key]=fit[1][:,0]
                
    yu.setpath('analysis_xJ')
    
    return cttejn_key2bare_B20

cttejn_key2bare_B20=yu.load_pkl_reg('cttejn_key2bare_B20')
if cttejn_key2bare_B20 is None:
    cttejn_key2bare_B20=run()
    yu.save_pkl_reg('cttejn_key2bare_B20',cttejn_key2bare_B20)

In [38]:
def run():
    ej_key2bare_B20at0={}
    
    colors=['r','g','b']
    for j in js_conn+js_disc:
        tftcphy = tftcphy_B20_conn if j in js_conn else tftcphy_B20_discq if j in js_discq else tftcphy_B20_gluon if j in js_gluon else 1/0
        c1s=['unequal','all','equal'] if j in js_conn else ['unequal']
        
        fig,axs=yu.getFigAxs(len(enss),1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey='row')
        fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
        yu.addRowHeader(axs,enss)
        yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])
        
        for axr in axs:
            for ax in axr:
                yu.addRefLine(ax,0)

        for ic1,c1 in enumerate(c1s):
            key2bare={cttejn[3:]:cttejn_key2bare_B20[cttejn] for cttejn in cttejn_key2bare_B20.keys() if cttejn[:3]==(c1,tftcphy[0],tftcphy[1])}
            for iens,ens in enumerate(enss):
                ax=axs[iens,0]            
                xunit=1; yunit=1
                
                if j in js_conn:
                    Zqq='Zqq^s' if j in ['j+;conn'] else 'Zqq'
                    mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                    Z=ens2RCs_me[ens][f'{Zqq}({mn})']
                else:
                    Z=1
                
                n2qpp1s=[key[-1] for key in key2bare.keys() if key[0]==ens and key[1]==j]
                Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
                ys=np.array([key2bare[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T
                
                color=colors[ic1]
                
                mean,err=yu.jackme(ys*Z)
                plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                
                ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
                axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
                
                
                if j in ['j-;conn']:
                    Q2cut=1
                    doFit=yu.doFit_dipole
                    fitfunc=yu.fitfunc_dipole
                else:
                    Q2cut=0.5
                    doFit=yu.doFit_const
                    fitfunc=yu.fitfunc_const
                
                inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
                Q2s=Q2s[inds]
                ys=ys[:,inds]
                
                pars_jk,chi2_jk,Ndof=doFit(Q2s,ys,corrQ=False) if doFit!=yu.doFit_const else doFit(ys,corrQ=False) 
                
                xs_plt=np.arange(0,Q2cut+1e-5,0.05)
                y_plt=np.array([fitfunc(xs_plt,*pars) for pars in pars_jk])
                mean,err=yu.jackme(y_plt*Z)
                plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                
                axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
                axs[iens,ic1+1].legend()
                
                axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
                axs[iens,-1].set_xlim([-5,5])
                axs[iens,-1].legend()
                
                if c1=='unequal' and j in ['j+;conn']+js_disc:
                    t=y_plt[:,0]
                    ej_key2bare_B20at0[(ens,j)]=t
                if c1=='all' and j=='j-;conn':
                    factor=ens2RCs_np_me[ens][f'Zqq(mu=nu)']/ens2RCs_np_me[ens][f'Zqq(mu!=nu)']
                    t=y_plt[:,0]*factor
                    ej_key2bare_B20at0[(ens,j)]=t
                
        yu.finalizePlot(f'Q2_mulc1_B20/{j}',mkdirQ=True)
        
    return ej_key2bare_B20at0

ej_key2bare_B20at0=yu.load_pkl_reg('ej_key2bare_B20at0')
if ej_key2bare_B20at0 is None:
    ej_key2bare_B20at0=run()
    yu.save_pkl_reg('ej_key2bare_B20at0',ej_key2bare_B20at0)

# C20

In [39]:
def run():
    cttejn_key2bare_C20={}

    yu.setpath('analysis_C20')
    def decodeTask(task):
        n2qpp1,ff,j=task.split('_')
        j=j.replace(',',';')
        n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
        return (n2qpp1,ff,j)
    path='data_aux/dat_ignore/analysis_C20_conn_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_conn:
            for c1 in ['all','unequal','equal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_2st')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_C20[key]=fit[1][:,0]
                        
    yu.setpath('analysis_C20')
    path='data_aux/dat_ignore/analysis_C20_disc_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_disc:
            for c1 in ['unequal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_const')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_C20[key]=fit[1][:,0]
                
    yu.setpath('analysis_xJ')
    
    return cttejn_key2bare_C20

cttejn_key2bare_C20=yu.load_pkl_reg('cttejn_key2bare_C20')
if cttejn_key2bare_C20 is None:
    cttejn_key2bare_C20=run()
    yu.save_pkl_reg('cttejn_key2bare_C20',cttejn_key2bare_C20)

In [40]:
def run():
    ej_key2bare_C20at0={}
    
    colors=['r','g','b']
    for j in js_conn+js_disc:
        tftcphy = tftcphy_C20_conn if j in js_conn else tftcphy_C20_discq if j in js_discq else tftcphy_C20_gluon if j in js_gluon else 1/0
        c1s=['unequal','all','equal'] if j in js_conn else ['unequal']
        
        fig,axs=yu.getFigAxs(len(enss),1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey='row')
        fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
        yu.addRowHeader(axs,enss)
        yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])
        
        for axr in axs:
            for ax in axr:
                yu.addRefLine(ax,0)

        for ic1,c1 in enumerate(c1s):
            key2bare={cttejn[3:]:cttejn_key2bare_C20[cttejn] for cttejn in cttejn_key2bare_C20.keys() if cttejn[:3]==(c1,tftcphy[0],tftcphy[1])}
            for iens,ens in enumerate(enss):
                ax=axs[iens,0]            
                xunit=1; yunit=1
                
                if j in js_conn:
                    Zqq='Zqq^s' if j in ['j+;conn'] else 'Zqq'
                    mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                    Z=ens2RCs_me[ens][f'{Zqq}({mn})']
                else:
                    Z=1
                
                n2qpp1s=[key[-1] for key in key2bare.keys() if key[0]==ens and key[1]==j]
                Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
                ys=np.array([key2bare[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T
                
                color=colors[ic1]
                
                mean,err=yu.jackme(ys*Z)
                plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                
                ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
                axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
                
                
                if j in ['j+;conn']:
                    Q2cut=1
                    doFit=yu.doFit_dipole
                    fitfunc=yu.fitfunc_dipole
                else:
                    Q2cut=0.5
                    doFit=yu.doFit_const
                    fitfunc=yu.fitfunc_const
                
                inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
                Q2s=Q2s[inds]
                ys=ys[:,inds]
                
                pars_jk,chi2_jk,Ndof=doFit(Q2s,ys,corrQ=False) if doFit!=yu.doFit_const else doFit(ys,corrQ=False) 
                
                xs_plt=np.arange(0,Q2cut+1e-5,0.05)
                y_plt=np.array([fitfunc(xs_plt,*pars) for pars in pars_jk])
                mean,err=yu.jackme(y_plt*Z)
                plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                
                axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
                axs[iens,ic1+1].legend()
                
                axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
                axs[iens,-1].set_xlim([-5,5])
                axs[iens,-1].legend()
                
                if c1=='unequal' and j in ['j+;conn']+js_disc:
                    t=y_plt[:,0]
                    ej_key2bare_C20at0[(ens,j)]=t
                if c1=='all' and j=='j-;conn':
                    factor=ens2RCs_np_me[ens][f'Zqq(mu=nu)']/ens2RCs_np_me[ens][f'Zqq(mu!=nu)']
                    t=y_plt[:,0]*factor
                    ej_key2bare_C20at0[(ens,j)]=t
                
        yu.finalizePlot(f'Q2_mulc1_C20/{j}',mkdirQ=True)
        
    return ej_key2bare_C20at0

ej_key2bare_C20at0=yu.load_pkl_reg('ej_key2bare_C20at0')
if ej_key2bare_C20at0 is None:
    ej_key2bare_C20at0=run()
    yu.save_pkl_reg('ej_key2bare_C20at0',ej_key2bare_C20at0)

# plots

In [41]:
PDF_SET_1='JAM22-PDF_proton_nlo'
PDF_SET_2='NNPDF40_nnlo_as_01180'

j2me_1=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET_1}.pkl')
j2me_2=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET_2}.pkl')
j2color={'jq':'purple','jg':'cyan','jtot':'grey','ju':'r','jd':'g','js':'b','jc':'orange'}
j2label={'jq':'q','jg':'g','jtot':'N','ju':'u','jd':'d','js':'s','jc':'c'}
j2fmt={'jq':'d','jg':'s','jtot':'o','ju':'^','jd':'v','js':'<','jc':'>'}

def addexp(ax0,ax1):
    ax=ax0
    for j in ['jtot','jq','jg']:
        m,e=j2me_1[j]
        ax.errorbar(0.0002,m,e,color=j2color[j],mfc=None,fmt=j2fmt[j],markersize=4)
        m,e=j2me_2[j]
        ax.errorbar(0.0004,m,e,color=j2color[j],mfc='white',fmt=j2fmt[j],markersize=4)
    ax=ax1
    for j in ['ju','jd','js','jc']:
        m,e=j2me_1[j]
        ax.errorbar(0.0002,m,e,color=j2color[j],mfc=None,fmt=j2fmt[j],markersize=4)
        m,e=j2me_2[j]
        ax.errorbar(0.0004,m,e,color=j2color[j],mfc='white',fmt=j2fmt[j],markersize=4)

In [42]:
supjackQ=True

stout=10
key2bare_A20=ej_key2bare_A20at0.copy()
yum.extendBare_avgx(key2bare_A20)
key2phy_A20=yum.convert_key2phy_stout(yum.bareRC2phy_avgx_np(key2bare_A20,ens2RCs_np,supjackQ=supjackQ),stout)

key2bare_B20=ej_key2bare_B20at0.copy()
yum.extendBare_avgx(key2bare_B20)
key2phy_B20=yum.convert_key2phy_stout(yum.bareRC2phy_avgx_np(key2bare_B20,ens2RCs_np,supjackQ=supjackQ),stout)

if supjackQ:
    key2phy={}
    for key in key2phy_A20.keys():
        key2phy[key]=(key2phy_A20[key]+key2phy_B20[key])/2
    key2phy_J=key2phy

    key2phy_DeltaSigmaBy2=None
    key2phy_L=None
else:
    key2bare={}
    for key in key2bare_A20.keys():
        key2bare[key]=(key2bare_A20[key]+key2bare_B20[key])/2
    key2bare_J=key2bare
    yum.extendBare_avgx(key2bare_J)
    key2phy_J=yum.convert_key2phy_stout(yum.bareRC2phy_avgx_np(key2bare_J,ens2RCs_np,supjackQ=supjackQ),stout)
    
    key2phy_DeltaSigmaBy2=None
    key2phy_L=None

yu.save_pkl_reg(f'key2phy_A20',key2phy_A20)
yu.save_pkl_reg(f'key2phy_B20',key2phy_B20)
yu.save_pkl_reg(f'key2phy_J',key2phy_J)

In [43]:
cases=['avgx','B20','J']
for case in cases:
    key2phy={'avgx':key2phy_A20,'B20':key2phy_B20,'J':key2phy_J}[case]

    list_dic=[{
        'key2phy':key2phy,
        # 'key2phy_pre':key2phy
    }]
    fig,axs=yum.makePlot_a2dependence_avgx(list_dic,case=case)
    # addexp(axs[0,0],axs[1,0])
    
    if case in ['avgx']:
        axs[1,0].set_ylim([-0.1,0.55])
    if case in ['B20']:
        axs[0,0].set_ylim([-0.3,0.3])
        axs[1,0].set_ylim([-0.19,0.19])

    yu.finalizePlot(f'a2dep_{case}')
    
cases=['avgx']
for case in cases:
    key2phy={'avgx':key2phy_A20,'B20':key2phy_B20,'J':key2phy_J}[case]

    list_dic=[{
        'key2phy':key2phy,
        # 'key2phy_pre':key2phy
    }]
    fig,axs=yum.makePlot_a2dependence_avgx(list_dic,case=case)
    addexp(axs[0,0],axs[1,0])
    
    if case in ['avgx']:
        axs[1,0].set_ylim([-0.1,0.55])

    yu.finalizePlot(f'a2dep_{case}_withPhe')

In [44]:
# app='Extrapolated'
# yu.save_pkl_reg(f'key2phy_A20_{app}',key2phy_A20)

In [45]:
def run(fig,axs,color,app=None):
    ax=axs[0,0]
    case='avgx'
    ce='MA'
    key2phy=key2phy_A20
    
    if app is not None:
        key2phy=yu.load_pkl_reg(f'key2phy_A20_{app}')

    j='jv1'
    fmt='s'

    # ax.set_ylim([-0.1,0.5])

    m,e=j2me_1[j]
    ax.errorbar(0.0002,m,e,color=color,mfc=None,fmt=fmt,markersize=4)
    m,e=j2me_2[j]
    ax.errorbar(0.0004,m,e,color=color,mfc='white',fmt=fmt,markersize=4)

    caselabel={'avgx':r'\langle \mathrm{x} \rangle', 'B20':r'B20', 'J':r'J'}[case]
    mean,err=yu.jackme(key2phy[(f'a=#_{ce}',j)])
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)

    label=rf'${caselabel}_{{u-d}}=$'+yu.un2str(mean[0],err[0],forceResult=1)
    for iens,ens in enumerate(enss):
        plt_x=yu.ens2a[ens]**2; plt_y,plt_yerr=yu.jackme(key2phy[(ens,j)])
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=label if iens==0 else None)
    ax.legend()

case='avgx'
j='jv1'

fig,axs=yu.getFigAxs(1,1,Lrow=4,Lcol=6)
ax=axs[0,0]
ax.set_xlim([0,yum.lat_a2s_plt[-1]])
ax.set_xticks([0,0.003,0.006])
ax.set_ylim([0.135,0.205])
run(fig,axs,'r')
yu.finalizePlot(f'a2dep_{case};{j}_withPhe')

# fig,axs=yu.getFigAxs(1,1,Lrow=4,Lcol=6)
# ax=axs[0,0]
# ax.set_xlim([0,yum.lat_a2s_plt[-1]])
# ax.set_xticks([0,0.003,0.006])
# ax.set_ylim([0.135,0.205])
# run(fig,axs,'r',app='Extrapolated')
# yu.finalizePlot(f'a2dep_{case};{j}_withPhe_Extrapolated')

# fig,axs=yu.getFigAxs(1,1,Lrow=4,Lcol=6)
# ax=axs[0,0]
# ax.set_xlim([0,yum.lat_a2s_plt[-1]])
# ax.set_xticks([0,0.003,0.006])
# ax.set_ylim([0.135,0.205])
# run(fig,axs,'b',app='0Q2')
# yu.finalizePlot(f'a2dep_{case};{j}_withPhe_0Q2')
    
# fig,axs=yu.getFigAxs(1,1,Lrow=4,Lcol=6)
# ax=axs[0,0]
# ax.set_xlim([0,yum.lat_a2s_plt[-1]])
# ax.set_xticks([0,0.003,0.006])
# ax.set_ylim([0.135,0.205])
# run(fig,axs,'r',app='Extrapolated')
# run(fig,axs,'b',app='0Q2')
# yu.finalizePlot(f'a2dep_{case};{j}_withPhe')

In [46]:
print('now@a2=0','')
g=cttejn_key2bare_A20[('all',0.6,0.2,'b','j-;conn',(0,0,0))]
Z=ens2RCs_np_me['b']['Zqq(mu=nu)']
print('now@b',yu.jackme_un2str(g),Z,yu.jackme_un2str(g*Z))
print('paper','0.149(16)','1.151','0.171(18)')
print('phe',yu.un2str(*j2me_1['jv1']),yu.un2str(*j2me_2['jv1']))

now@a2=0 
now@b 0.151(13) 1.1167 0.169(14)
paper 0.149(16) 1.151 0.171(18)
phe 0.1542(35) 0.1533(41)


In [47]:
for case in ['avgx','J']:
    key2phy={'avgx':key2phy_A20,'B20':key2phy_B20,'J':key2phy_J,'DeltaSigmaBy2':key2phy_DeltaSigmaBy2,'L':key2phy_L}[case]
    fig,axs=yum.makePlot_percentBar(key2phy,case=case)
    yu.finalizePlot(f'bar_{case}')
    
for case in ['avgx']:
    key2phy={'avgx':key2phy_A20,'B20':key2phy_B20,'J':key2phy_J,'DeltaSigmaBy2':key2phy_DeltaSigmaBy2,'L':key2phy_L}[case]
    fig,axs=yum.makePlot_percentBar(key2phy,case=case)
    ax=axs[0,0]
    
    if case in ['J']:
        ax.set_ylim([0,0.6])
    
    if case in ['avgx']:
        ax.set_ylim([0,1.2])

        j2color={'jq':'purple','jg':'cyan','jtot':'grey','ju':'r','jd':'g','js':'b','jc':'orange'}
        j2label={'jq':'q','jg':'g','jtot':'N','ju':'u','jd':'d','js':'s','jc':'c'}
        j2fmt={'jq':'d','jg':'s','jtot':'o','ju':'^','jd':'v','js':'<','jc':'>'}

        PDF_SET='JAM22-PDF_proton_nlo'
        j2me=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET}.pkl')
        js=['ju','jd','js','jc','jq','jg','jtot']
        for ij,j in enumerate(js):
            m,e=j2me[j]
            ax.errorbar(ij+0.3,m,e,color=j2color[j],markersize=5,label=None if j!='jtot' else 'JAM22')
            
        PDF_SET='NNPDF40_nnlo_as_01180'
        j2me=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET}.pkl')
        js=['ju','jd','js','jc','jq','jg','jtot']
        for ij,j in enumerate(js):
            m,e=j2me[j]
            ax.errorbar(ij+0.5,m,e,color=j2color[j],markersize=5,mfc='white',label=None if j!='jtot' else 'NNPDF4.0')

        ax.legend()
    yu.finalizePlot(f'bar_{case}_withPhe')

In [48]:
# combined plot for Lattice Proceeding of Dina

fig,axs=yu.getFigAxs(2,3,sharex='col',gridspec_kw={'hspace': 0})

fontsize=32
fontsize_bar=18
fontsize_label=32

if True:
    plotwhich='both'; noNumberQ=True; noCEonPreQ=False; ce='MA'
    lat_a2s_plt=yum.lat_a2s_plt
    
    for icol, case, key2phy in zip([0,1],['avgx','J'],[key2phy_A20,key2phy_J]):
        caselabel={'avgx':r'\langle \mathrm{x} \rangle', 'B20':r'B', 'J':r'J'}[case]
        extralabel={'avgx':'','B20':'20;','J':''}[case]
        
        if case == 'avgx':
            if plotwhich in ['both']:
                ax=axs[0,icol]
                ax.set_ylim([0,1.45])
                ax.set_yticks(np.arange(0.2,1.3,0.2))
                ax=axs[1,icol]
                ax.set_ylim([-0.05,0.54])
                ax.set_yticks(np.arange(0,0.5,0.1))
                for icol in [icol]:
                    axs[0,icol].axhline(1,color='black',ls='--',marker='',
                                            linewidth=2)
                    # axs[0,icol].axhline(0,color='black',ls='dotted',marker='')
                    axs[1,icol].axhline(0,color='black',ls='dotted',marker='',
                                            linewidth=2)
                
        if case =='J':
            if plotwhich in ['both']:
                ax=axs[0,icol]
                ax.set_ylim([0.,0.75])
                ax.set_yticks(np.arange(0.2,0.7,0.2))
                ax=axs[1,icol]
                ax.set_ylim([-0.05,0.34])
                ax.set_yticks(np.arange(0,0.3,0.1))
                for icol in [icol]:
                    # axs[0,icol].axhline(0,color='black',ls='dotted',marker='')
                    axs[0,icol].axhline(0.5,color='black',ls='--',marker='',
                                            linewidth=2)
                    axs[1,icol].axhline(0,color='black',ls='dotted',marker='',
                                            linewidth=2)
                    axs[1,icol].axhline(0.5,color='black',ls='--',marker='',
                                            linewidth=2)
        
        if plotwhich in ['both']:
            ax=axs[0,icol]
            ax.set_xlim([0,lat_a2s_plt[-1]])
            ax.set_xticks([0,0.003,0.006])
            ax.set_ylabel(rf'${caselabel}_{{{extralabel}q,g}}$',size=fontsize_label)

            ax=axs[1,icol]
            ax.set_ylabel(rf'${caselabel}_{{{extralabel}q}}$',size=fontsize_label)
                
            for icol in [icol]:
                axs[1,icol].set_xlabel(r'$a^2$ [fm$^2$]',size=fontsize_label)
            
        for icol in [icol]:
            dic={'key2phy':key2phy}
            def setParameter(default,key):
                return dic[key] if key in dic else default
            
            key2phy=dic['key2phy']
            key2phy_pre=setParameter(None,'key2phy_pre')
            
            if key2phy_pre is not None:
                keys=list(key2phy_pre)
                enss_pre=yu.removeDuplicates([ens for ens,j in keys if 'a=#' not in ens])
            
            keys=list(key2phy)
            enss=yu.removeDuplicates([ens for ens,j in keys if 'a=#' not in ens])
            
            def get(ens,j):
                return key2phy[(ens,j)]
            def get_pre(ens,j):
                return key2phy_pre[(ens,j)]
            
            j2color={'jq':'purple','jg':'cyan','jtot':'grey','ju':'r','jd':'g','js':'b','jc':'orange',\
                'jv1':'r','jv2':'g','jv3':'b'}
            j2label={'jq':'q','jg':'g','jtot':'N','ju':'u','jd':'d','js':'s','jc':'c',\
                'jv1':'u-d','jv2':'u+d-2s','jv3':'u+d+s-3c'}
            j2fmt={'jq':'d','jg':'s','jtot':'o','ju':'^','jd':'v','js':'<','jc':'>',\
                'jv1':'s','jv2':'d','jv3':'o'}
            

            ax=axs[0,icol]
            js=['jq','jtot','jg']
            for ij,j in enumerate(js):
                color=j2color[j]
                mean,err=yu.jackme(get(f'a=#_{ce}',j))
                label=rf'${caselabel}_{{{extralabel}{j2label[j]}}}=$'+yu.un2str(mean[0],err[0],forceResult=1)
                if noNumberQ:
                    label=rf'${caselabel}_{{{extralabel}{j2label[j]}}}$'
                for iens,ens in enumerate(enss):
                    plt_x=yu.ens2a[ens]**2+(ij-len(js)/2)*5e-10; plt_y,plt_yerr=yu.jackme(get(ens,j))
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=j2fmt[j],label=label if iens==0 else None, ms=12)
                    
                    if key2phy_pre is None or ens not in enss_pre:
                        continue
                    plt_x=yu.ens2a[ens]**2+0.0001; plt_y,plt_yerr=yu.jackme(get_pre(ens,j))
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=j2fmt[j],mfc='white')
                        
                mean,err=yu.jackme(get(f'a=#_{ce}',j))
                x=lat_a2s_plt; ymin=mean-err; ymax=mean+err
                ax.plot(x,mean,color=color,linestyle='--',marker='')
                ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
                
                if not noCEonPreQ and key2phy_pre is not None:
                    plt_x=0.0004; plt_y,plt_yerr=yu.jackme(get_pre(f'a=#_{ce}',j)[:,0])
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=j2fmt[j],mfc='white')
                
            ax.legend(fontsize=fontsize,ncol=3,loc='upper left',
                          columnspacing=0.1,   # space between columns
    handletextpad=0.1,   # space between line/marker and text
    handlelength=1.0,    # shorten legend line
    labelspacing=0.1,    # vertical spacing
    borderpad=0.2)

            ax=axs[1,icol] if plotwhich in ['both'] else axs[0,icol]
            js=['ju','jd','js','jc']
            for ij,j in enumerate(js):
                color=j2color[j]
                mean,err=yu.jackme(get(f'a=#_{ce}',j))
                label=rf'${caselabel}_{{{extralabel}{j2label[j]}}}=$'+yu.un2str(mean[0],err[0],forceResult=1)
                if noNumberQ:
                    label=rf'${caselabel}_{{{extralabel}{j2label[j]}}}$'
                for iens,ens in enumerate(enss):
                    plt_x=yu.ens2a[ens]**2+(ij-len(js)/2)*5e-10; plt_y,plt_yerr=yu.jackme(get(ens,j))
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=j2fmt[j],label=label if iens==0 else None, ms=12)
                    if key2phy_pre is None or ens not in enss_pre:
                        continue
                    plt_x=yu.ens2a[ens]**2+0.0001; plt_y,plt_yerr=yu.jackme(get_pre(ens,j))
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=j2fmt[j],mfc='white',markersize=24)

                mean,err=yu.jackme(get(f'a=#_{ce}',j))
                x=lat_a2s_plt; ymin=mean-err; ymax=mean+err
                ax.plot(x,mean,color=color,linestyle='--',marker='')
                ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
                
                if not noCEonPreQ and key2phy_pre is not None:
                    plt_x=0.0004; plt_y,plt_yerr=yu.jackme(get_pre(f'a=#_{ce}',j)[:,0])
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=j2fmt[j],mfc='white')
                    
            ax.legend(fontsize=fontsize,ncol=4,loc='upper left',
                          columnspacing=0.1,   # space between columns
    handletextpad=0.1,   # space between line/marker and text
    handlelength=1.0,    # shorten legend line
    labelspacing=0.1,    # vertical spacing
    borderpad=0.2)
        
    # =============================
    icol=2
    noNumbersOnBar=False
    for irow,case,key2phy in zip([0,1],['avgx','J'],[key2phy_A20,key2phy_J]):
        caselabel={'avgx':r'\langle \mathrm{x} \rangle', 'B20':r'B', 'J':r'J', \
                    'DeltaSigmaBy2':r'\frac{1}{2}\Delta\Sigma', 'L':r'L'}[case]
        extralabel={'B20':'20;'}[case] if case in ['B20'] else ''
        
        valRef=1 if case in ['avgx'] else 0.5
        
        if case in ['avgx','J','DeltaSigmaBy2','L']:
            ax=axs[irow,icol]
            ax.axhline(valRef,color='black',ls='--',marker='',
                                            linewidth=2)
        if case in ['DeltaSigmaBy2','L']:
            ax.axhline(0,color='black',ls='--',marker='',
                                            linewidth=2)
        
        if case in ['DeltaSigmaBy2','L']:
            labelpad={'DeltaSigmaBy2':-21,'L':-17}[case]
        else:
            labelpad=None
        if case in ['DeltaSigmaBy2']:
            ax.set_ylabel(rf'${caselabel}_{{{extralabel}q}}$',labelpad=labelpad,fontsize=fontsize_label)
        else:
            ax.set_ylabel(rf'${caselabel}_{{{extralabel}q,g}}$',labelpad=labelpad,fontsize=fontsize_label)
        
        ax.set_ylim({'avgx':[0,1.35],'J':[0,0.65],'DeltaSigmaBy2':[-0.3,0.55],'L':[-0.3,0.55]}[case])
        if irow==0:
            ax.set_yticks(np.arange(0.2,1.1,0.2))
        
        if case in ['avgx','J']:
            js = ['ju','jd','js','jc','jq','jg','jtot']
            # labels = [r'$u^+$', r'$d^+$', r'$s^+$', r'$c^+$', r'$\sum_{q^+ = u,d,s,c}$', r'$g$', 'Total']
            labels = [r'$u$', r'$d$', r'$s$', r'$c$', r'$q$', r'$g$', 'Total']
            colors = np.array(['r','g','b','orange','purple','cyan','grey'])
            
            inds_conn=[0,1,4,6]
            js_conn=[js[ind] for ind in inds_conn]

            x = np.arange(len(labels))
            ax.set_xticks(x)
            ax.set_xticklabels(labels,size=fontsize_label-4)
            ax.set_xlim([x[0]-3/4,x[-1]+3/4])
                        
            factor=valRef
            mes_conn=[yu.jackme(key2phy[('a=#_MA',f'{j};conn')][:,0]) for j in js_conn]
            ax.bar(x[inds_conn], [ele[0] for ele in mes_conn], capsize=5, color=colors[inds_conn], alpha=0.8, width=0.3, edgecolor='grey')

            mes=[yu.jackme(key2phy[('a=#_MA',f'{j}')][:,0]) for j in js]
            bars=ax.bar(x, [ele[0] for ele in mes], yerr=[ele[1] for ele in mes], capsize=5, color=colors, alpha=0.2, width=0.5, edgecolor='grey')
            func=lambda res:'{:.1f}({:.1f})%'.format(res[0]*100/factor,res[1]*100/factor)
            percentages=[func(ele) for ele in mes]
            
            if not noNumbersOnBar:
                for i, (bar, pct) in enumerate(zip(bars, percentages)):
                    height = bar.get_height()
                    height = 0.01 if height < 0.2 else height - 0.1
                    if bar.get_height()<-0.01:
                        height = bar.get_height() + 0.1
                    ax.text(bar.get_x() - 0.1, height, pct,
                            ha='center', va='bottom' if bar.get_height()>-0.01 else 'top', fontsize=fontsize_bar,  rotation=90)
             
for axr in axs:       
    for ax in axr:  # if you have multiple subplots
        for spine in ax.spines.values():
            spine.set_linewidth(2)

yu.finalizePlot('combined_avgx_J',closeQ=True)